In this script we do RAG with a .txt file and .pdf.

What do we need?

1. Text Spliting by chunking -> LLMs have context window, which limits the amount of words it can process in a single time (in order of thousands).

2. Embedding -> the way LLM convert your prompt to high dimensionality vectors

3. Vector databases -> store the embedded chunks. Ex: InMemory, FAISS, ChromaDB for small/medium projects

As you can see below, we're going to use the new version Langchain 1.0, with some changes to its implementation and importing.

In [ ]:
from langchain_google_genai import ChatGoogleGenerativeAI, GoogleGenerativeAIEmbeddings
from langchain_text_splitters import RecursiveCharacterTextSplitter
from langchain_community.vectorstores import InMemoryVectorStore

from langchain.agents import create_agent
from langchain.tools import tool

from langchain_community.document_loaders import PyPDFLoader

import os
from dotenv import load_dotenv

load_dotenv(override=True) #override to get changes in the .env file

API_KEY = os.getenv('API_KEY')

In [2]:
llm = ChatGoogleGenerativeAI(
    model="gemini-2.0-flash-lite", # smallest model
    google_api_key=API_KEY,
)

First, let's get the raw text from the .pdf.

In [3]:
loader = PyPDFLoader(file_path="./data/scientific_initiation_report.pdf")

docs = loader.load()
text = docs[0].page_content
print(text[:94])

Scientific Initiation Report – PICME Scholarship – Semester 25.1 
Scholar: Bruno Véras Schulz 


We must split the text from .txt and .pdf into chunks, since LLMs have context window, which limits the amount of words it can process in a single time.

In [4]:
text_splitter = RecursiveCharacterTextSplitter(chunk_size=500, chunk_overlap=50)

# .txt chunks
chunks = text_splitter.split_text(open("data/curriculum.txt", encoding="utf-8").read())

# extending .pdf chunks into the chunks list
chunks.extend(text_splitter.split_text(text))

Let's create the Embedding object that convert the chunks to vectors, and then create a vector database using InMemoryVectorStore (runtime) to storage the vectors.

In [5]:
emb = GoogleGenerativeAIEmbeddings(
    model="models/gemini-embedding-001",
    google_api_key=API_KEY
    )

In [6]:
chunks

["Bruno's curriculum\n\n\n1. Introduction\n\nComputer Science undergraduate student seeking an internship to gain professional experience and foster career \ndevelopment. Throughout my academic journey, I have been awarded over 30 medals in Olympiads across a wide range of \nfields, from Mathematics to Economics. Proactive, communicative, and a fast learner, I am looking for a company that values \ninterpersonal skills such as teamwork and leadership.\n\n\n2. Education",
 '2. Education\n\nB.Sc. in Computer Science – Federal University of Rio de Janeiro (UFRJ)\nGPA: 9.6/10 | Expected Graduation: 2027.2\n\nExchange Semester – Technical University of Munich',
 'Exchange program with scholarship at the best university in the European Union and 27th in the world. Taking the opportunity \nto deepen my knowledge in my field of interest: Data Science. \nCourses taken: Business Analytics and Machine Learning, Introduction to Deep Learning, Fundamentals of Artificial \nIntelligence, Visual Data 

In [7]:
vector_store = InMemoryVectorStore.from_texts(texts=chunks, embedding=emb)
retriever = vector_store.as_retriever()

Now creating a tool is needed to retrieve info from the vector database.

In [8]:
@tool()
def retrieve_context(query: str):
    """Retrieve information to help answer a query."""
    retrieved_chunks = vector_store.similarity_search(query, k=5)

    return retrieved_chunks

tools = [retrieve_context]

In [9]:
agent = create_agent(
    model=llm,
    system_prompt="You have access to a tool that retrieves information about Bruno. You have his curriculum and a Scientific Initiation Report.",
    tools=tools
)

We should be able to do prompts with RAG, let's test:

In [15]:
output = agent.invoke({"messages": [{"role": "user", "content": "What is Bruno's Education?"}]})

In [21]:
output["messages"][-1].content

'Bruno is a Computer Science undergraduate student at the Federal University of Rio de Janeiro (UFRJ), with an expected graduation in 2027. He also did an exchange semester at the Technical University of Munich.'

In [12]:
def response(agent, query):
    return (agent.invoke({"messages": [{"role": "user", "content": query}]}))["messages"][-1].content

In [25]:
response(agent, "Give me a short description of Bruno's projects.")

"I am sorry, I cannot provide a description of Bruno's projects. I can only retrieve information about Bruno's curriculum."

In [26]:
response(agent, "Give me a short description of Bruno's projects that are inside his curriculum text.")

'Bruno has been a Back-end Developer at UFRJ’s Junior Consulting and Microinformatics Company, where he contributed to a full project, gaining experience across the entire software development lifecycle. He also participated in the Extension Project – InformAÇÃO, where he developed training material on Critical Data Literacy for K-12 teachers.'

Trying a smarter model to see if our responses are better:

In [10]:
llm = ChatGoogleGenerativeAI(
    model="gemini-2.5-flash",
    google_api_key=API_KEY,
)

agent = create_agent(
    model=llm,
    system_prompt="You have access to a tool that retrieves information about Bruno's curriculum.",
    tools=tools
)

In [32]:
res = response(agent, "Give me a short description of Bruno's projects.")

In [36]:
res[0]["text"]

'Bruno has worked on several projects, including an Extension Project called "InformAÇÃO" where he developed training material on Critical Data Literacy for K-12 teachers. He also completed a "Retail Store Demand Forecasting Model" as a final project for a Data Science Bootcamp, involving data collection, preprocessing, exploratory analysis, feature engineering, and machine learning model development. Additionally, Bruno developed a "Database-Integrated Website" using Python, SQLAlchemy for a MySQL database, and the Flask framework for the website.'

In [13]:
res = response(agent, "Give me a short description about Bruno's Scientific Initiation.")

In [14]:
res[0]["text"]

'Bruno has been involved in Scientific Initiation for several years. He was a scholarship holder for 5 years in the PIC program due to his achievements in the Brazilian Mathematics Olympiad, and is also a scholarship holder in the PICME program. Since August 2023, he has been researching Differential Algebra under the supervision of Professor Collier. This research involves studying the subject using notes that will become a book, reviewing and correcting them, and exploring differential equation systems using the Rosenfeld–Gröbner decomposition algorithm. He has presented his work at four undergraduate research events and is currently submitting a paper.'